# 🎭 Yakshagana Vesha Classification with "Not Yakshagana" Detection (v4, photos only)

This notebook trains and compares **four ImageNet-pretrained CNNs** (ResNet50, EfficientNetB0, MobileNetV2, DenseNet121) to predict the **Vesha type** of a Yakshagana performer photo (e.g. Rajavesha, StreeVesha, RakshasaVesha).

### What is new compared with v2
**"Not Yakshagana" detection.** The model has one extra class. If you upload a Bharatanatya photo (or a crowd, a temple, a selfie, a film still...) it answers **"This photo is not related to Yakshagana"** instead of forcing it into a Vesha type.

> ⚠️ **Honest notes**
> * This notebook contains **no pre-computed results**. Every number appears only after *you* run it on *your* dataset.
> * The "Not Yakshagana" class only works if you **give it examples**: add a folder of non-Yakshagana photos (see below). Without it the model cannot reject anything.

### Before you start
* **Runtime → Change runtime type → T4 GPU**, then run the cells **top to bottom**.
* Dataset layout: **the top-level folder name is the label**. Add one extra folder for non-Yakshagana images:

```
dataset/
├── Rajavesha/        img001.jpg, img002.jpg, ...
├── StreeVesha/       ...
├── RakshasaVesha/    ...
└── NotYakshagana/    bharatanatya/..., kathak/..., crowd/..., temple/..., portraits/..., random/...
```
Sub-folders inside any class folder are fine (the character folder is simply **ignored**). Folder names `NotYakshagana`, `Other`, `NonYakshagana`, `Unknown` ... are all recognised as the "not Yakshagana" class.

**What to put in `NotYakshagana/`** (this decides how good the rejection is):
* **Hard negatives (most important):** Bharatanatya, Kathak, Kuchipudi, Kathakali, Mohiniyattam, Dollu Kunitha, Bhootha Kola, Theyyam, drama/stage plays, folk dances. They look "colourful and traditional" like Yakshagana, so the model must see them.
* **Easy negatives:** audience/crowd, temples, empty stages, landscapes, selfies, street scenes, objects.
* Aim for **at least as many images as the average Vesha class**, ideally 2-3x (capped at `MAX_OTHER_RATIO` x the average Vesha size).

### Notebook map
| Part | What happens |
|---|---|
| **A** | Install, imports, configuration, Google Drive |
| **B** | Reusable functions (model builder, metrics, "Not Yakshagana" decision logic) |
| **C** | Load dataset, validate, clean, group, split 70/15/15 |
| **D** | Train 4 models, curves, model selection (validation), final test evaluation, **Yakshagana-vs-Not report** |
| **E** | Load saved models (works in a fresh session, no retraining) |
| **F** | Upload your own image: Vesha prediction **or** "not related to Yakshagana" |
| **G** | Optional low-light robustness test |

**New session and just want to predict?** Run Part A → Part B → Part E → Part F.


## Part A — Setup
### A1. Install dependencies
Colab already includes TensorFlow, NumPy, pandas, scikit-learn, Matplotlib and Pillow. We install only the extras and deliberately **do not upgrade TensorFlow**.

In [ ]:
!pip install -q seaborn tqdm

### A2. Imports

In [ ]:
import os, io, re, sys, json, time, random, shutil, zipfile, hashlib
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps, ImageEnhance
from tqdm.auto import tqdm
from IPython.display import display

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)
from sklearn.utils.class_weight import compute_class_weight

print('TensorFlow', tf.__version__, '| Keras', keras.__version__)

### A3. Configuration (the only cell you normally need to edit)
* `DATA_SOURCE`: `'zip'` uploads a ZIP now; `'drive'` reads a folder or ZIP from Google Drive; `'local'` uses a folder already on the Colab disk.
* **Epochs** are split into two phases: `EPOCHS_HEAD` (backbone frozen) and `EPOCHS_FINE` (last `FINE_TUNE_LAYERS` backbone layers unfrozen at a low learning rate). Early stopping may end each phase sooner.
* **GPU out-of-memory?** Lower `BATCH_SIZE` (try 16).
* **Quick smoke test:** `MODELS_TO_TRAIN = ['MobileNetV2']`, `EPOCHS_HEAD = 2`, `EPOCHS_FINE = 1`.

**New in v4:** `OTHER_CLASS_NAME` / `OTHER_ALIASES` say which folder is the "not Yakshagana" class, `OTHER_THRESHOLD` is the probability above which a photo is rejected (tune it in D6b).


In [ ]:
CONFIG = dict(
    SEED=42,

    # ---------- Data source ----------
    DATA_SOURCE='zip',                        # 'zip' | 'drive' | 'local'
    DRIVE_DATA_SUBPATH='Yakshagana/dataset',  # relative to My Drive; folder OR .zip (when DATA_SOURCE='drive')
    LOCAL_DATA_PATH='/content/dataset',       # used when DATA_SOURCE='local'
    DATASET_ROOT_OVERRIDE=None,               # set a path to skip automatic dataset-root detection
    COPY_DRIVE_FOLDER_TO_LOCAL=True,          # copying to the Colab disk makes training much faster
    GROUP_CSV=None,                           # optional CSV (columns: relpath, group) naming the performer/performance per image

    # ---------- Outputs ----------
    DRIVE_SAVE_SUBPATH='Yakshagana_vesha_models_v4',   # models + class mapping are saved here (inside My Drive)
    WORK_DIR='/content/yakshagana_work',

    # ---------- Dataset cleaning ----------
    MIN_IMAGES_PER_CLASS=20,                  # Vesha classes with fewer images are dropped
    MIN_IMAGE_SIDE=32,
    MAX_CACHE_SIDE=512,
    DROP_EXACT_DUPLICATES=True,
    NEAR_DUP_HAMMING=4,                       # dHash distance (0-64) under which two images are near-duplicates

    # ---------- Split ----------
    SPLIT=(0.70, 0.15, 0.15),                 # train / validation / test
    N_FOLDS=20,                               # 20 folds of 5% -> 14 train / 3 val / 3 test
    SPLIT_ATTEMPTS=20,

    # ---------- Training ----------
    MODELS_TO_TRAIN=['ResNet50', 'EfficientNetB0', 'MobileNetV2', 'DenseNet121'],
    IMG_SIZE=224,
    BATCH_SIZE=32,
    EPOCHS_HEAD=8,
    EPOCHS_FINE=12,
    LR_HEAD=1e-3,
    LR_FINE=1e-5,
    FINE_TUNE_LAYERS=30,
    DROPOUT=0.3,
    ES_PATIENCE=5,
    LR_PATIENCE=2,
    LR_FACTOR=0.3,
    MIN_LR=1e-7,
    USE_CLASS_WEIGHTS=True,                   # counteracts class imbalance
    FORCE_RETRAIN=False,                      # False: skip models whose weights are already saved

    # ---------- Prediction / uncertainty thresholds (heuristics; tune on validation data) ----------
    CONF_THRESHOLD=0.60,
    MARGIN_THRESHOLD=0.15,                    # top-1 minus top-2 probability
    ENTROPY_THRESHOLD=0.80,                   # normalised entropy (0 = certain, 1 = uniform)
    PREDICTION_MODEL=None,                    # None -> best model recorded in class_mapping.json

    # ---------- "Not Yakshagana" class (v3) ----------
    OTHER_CLASS_NAME='NotYakshagana',         # top-level folder holding non-Yakshagana images
    OTHER_ALIASES=['Other', 'Others', 'NonYakshagana', 'Not_Yakshagana', 'Unknown', 'Negative', 'NotRelated', 'Irrelevant'],
    MAX_OTHER_RATIO=3.0,                      # cap: NotYakshagana images <= ratio x average Vesha class size
    OTHER_THRESHOLD=0.50,                     # P(NotYakshagana) >= this -> "not related to Yakshagana" (tune in D6b)
    OTHER_WARN=0.25,                          # P(NotYakshagana) between WARN and THRESHOLD -> flagged as uncertain
)

### A4. Reproducible seeds and GPU check
All random generators are seeded. The **data split** is fully reproducible; GPU training can still differ slightly between runs (some GPU kernels are non-deterministic).

In [ ]:
SEED = cfg.SEED

def set_global_seed(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    try:
        keras.utils.set_random_seed(seed)
    except AttributeError:
        pass

set_global_seed()
IN_COLAB = 'google.colab' in sys.modules
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for g in gpus:
        try: tf.config.experimental.set_memory_growth(g, True)
        except Exception: pass
    print(f'✅ GPU available: {[g.name for g in gpus]}')
else:
    print('⚠️ No GPU found. Use Runtime → Change runtime type → T4 GPU.')
print('Running in Colab:', IN_COLAB)

### A5. Mount Google Drive
Used to **save trained models and the class mapping** (and optionally to read your dataset). If mounting fails, files go to the temporary Colab disk, which is **deleted when the session ends**.

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive'
DRIVE_MOUNTED = False
REQUIRE_DRIVE = True   # True: stop if Drive is not mounted, so models are never lost silently
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_MOUNTED = os.path.isdir(DRIVE_ROOT)
    except Exception as e:
        print('⚠️ Could not mount Google Drive:', e)

if DRIVE_MOUNTED:
    SAVE_DIR = os.path.join(DRIVE_ROOT, cfg.DRIVE_SAVE_SUBPATH)
else:
    SAVE_DIR = os.path.join(cfg.WORK_DIR, 'saved_models')
    print('⚠️ Drive not mounted: models will only be saved to the temporary Colab disk.')
os.makedirs(SAVE_DIR, exist_ok=True)
print('Models and mapping will be saved to:', SAVE_DIR)

if IN_COLAB and REQUIRE_DRIVE and not DRIVE_MOUNTED:
    raise RuntimeError('Google Drive is NOT mounted, so trained models would be deleted when the session ends. '
                       'Re-run this cell and allow access. (Set REQUIRE_DRIVE = False only if you will download the models yourself.)')
# Prove that saving really works before spending an hour training
_t = os.path.join(SAVE_DIR, '_write_test.txt')
open(_t, 'w').write('ok'); assert open(_t).read() == 'ok'; os.remove(_t)
print('✅ Save folder is writable:', SAVE_DIR)

## Part B — Reusable definitions
These cells only *define* functions; nothing is trained yet.

### B1. The model
```
image → augmentation (training only) → backbone preprocessing → pretrained CNN → pooling
      → Dense layer → softmax over Vesha types
```
* **Preprocessing** differs per backbone (ResNet50 caffe-style, MobileNetV2 [-1, 1], DenseNet121 torch-style, EfficientNetB0 built in). It sits *inside* the model, so training, evaluation and prediction are always consistent.
* **Augmentation** (flip, small rotation/zoom/shift, brightness and contrast jitter) is also inside the model and Keras activates it **only while training**. Validation, test and prediction images are never augmented.
* **Transfer learning, two phases:** (1) freeze the pretrained backbone, train the new layers; (2) unfreeze the top backbone layers (BatchNorm stays frozen) and fine-tune at a much smaller learning rate.

In [ ]:
from tensorflow.keras.applications import ResNet50, EfficientNetB0, MobileNetV2, DenseNet121
from tensorflow.keras.applications import resnet50, efficientnet, mobilenet_v2, densenet

BACKBONES = {
    'ResNet50':       dict(ctor=ResNet50,       preprocess=resnet50.preprocess_input),
    'EfficientNetB0': dict(ctor=EfficientNetB0, preprocess=efficientnet.preprocess_input),  # identity: scaling is inside the model
    'MobileNetV2':    dict(ctor=MobileNetV2,    preprocess=mobilenet_v2.preprocess_input),
    'DenseNet121':    dict(ctor=DenseNet121,    preprocess=densenet.preprocess_input),
}

def build_augmentation(seed=SEED):
    """Random augmentation. Keras applies these layers ONLY when training=True."""
    return keras.Sequential([
        layers.RandomFlip('horizontal', seed=seed),
        layers.RandomRotation(0.05, seed=seed),
        layers.RandomZoom(0.10, seed=seed),
        layers.RandomTranslation(0.05, 0.05, seed=seed),
        layers.RandomContrast(0.20, seed=seed),
        layers.RandomBrightness((-0.4, 0.2), value_range=(0, 255), seed=seed),  # includes darkening
    ], name='train_augmentation')

def build_model(name, n_classes, img_size=224, pretrained=True, dropout=0.3):
    """Return (model, backbone). Input: RGB images with pixel values 0-255."""
    if name not in BACKBONES:
        raise ValueError(f'Unknown model {name}. Choose from {list(BACKBONES)}')
    spec = BACKBONES[name]
    inputs = keras.Input(shape=(img_size, img_size, 3), name='image')
    x = build_augmentation()(inputs)
    x = layers.Lambda(spec['preprocess'], name='preprocess')(x)
    base = spec['ctor'](include_top=False, weights='imagenet' if pretrained else None,
                        input_shape=(img_size, img_size, 3))
    base.trainable = False                                   # phase 1: frozen
    x = base(x, training=False)                              # keeps BatchNorm in inference mode
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(dropout, name='drop_1')(x)
    x = layers.Dense(256, activation='relu', name='dense_hidden')(x)
    x = layers.Dropout(dropout, name='drop_2')(x)
    out = layers.Dense(n_classes, activation='softmax', name='vesha')(x)
    return keras.Model(inputs, out, name=f'{name}_vesha'), base

def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])

def set_fine_tuning(base, n_layers):
    """Unfreeze the last n_layers of the backbone (BatchNorm layers stay frozen)."""
    if n_layers <= 0:
        return
    base.trainable = True
    for layer in base.layers[:max(0, len(base.layers) - n_layers)]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

def make_callbacks():
    return [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=cfg.ES_PATIENCE, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=cfg.LR_FACTOR, patience=cfg.LR_PATIENCE,
                                          min_lr=cfg.MIN_LR, verbose=1),
    ]
print('Model builders defined.')

### B2. Saving and loading models and the class mapping
Weights are saved as `<Model>.weights.h5`, plus `class_mapping.json` (class names including the NotYakshagana class, image size, best model name). A saved model is rebuilt with the same code and the weights are loaded, so **no retraining is needed**.

In [ ]:
ARTIFACTS = {}      # class names, image size, best model name
MODEL_CACHE = {}    # models already loaded into memory

def artifacts_path():       return os.path.join(SAVE_DIR, 'class_mapping.json')
def model_weights_path(n):  return os.path.join(SAVE_DIR, f'{n}.weights.h5')
def meta_path(n):           return os.path.join(SAVE_DIR, f'{n}_meta.json')

def save_artifacts():
    a = ARTIFACTS
    payload = dict(classes=list(a['classes']),                 # model output order (Vesha types + optional NotYakshagana)
                   other_class=a.get('other_class'),           # name of the "not Yakshagana" class, or None
                   img_size=int(a['img_size']),
                   models=list(a.get('models', [])), best_model=a.get('best_model'), seed=cfg.SEED)
    with open(artifacts_path(), 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

def load_artifacts():
    path = artifacts_path()
    if not os.path.exists(path):
        raise FileNotFoundError(f'No class_mapping.json in {SAVE_DIR}. Train first (Parts C-D) or check cfg.DRIVE_SAVE_SUBPATH.')
    with open(path, encoding='utf-8') as f:
        p = json.load(f)
    classes = p.get('classes') or p.get('vesha_classes')       # v2 files used 'vesha_classes'
    ARTIFACTS.clear()
    ARTIFACTS.update(classes=classes, other_class=p.get('other_class'), img_size=p['img_size'],
                     models=p.get('models', []), best_model=p.get('best_model'))
    return ARTIFACTS

def get_model(name):
    """Rebuild the architecture and load saved weights (no retraining, no ImageNet download)."""
    if name in MODEL_CACHE:
        return MODEL_CACHE[name]
    wp = model_weights_path(name)
    if not os.path.exists(wp):
        raise FileNotFoundError(f'Weights not found: {wp}')
    model, _ = build_model(name, len(ARTIFACTS['classes']), ARTIFACTS['img_size'],
                           pretrained=False, dropout=cfg.DROPOUT)
    model.load_weights(wp)
    MODEL_CACHE[name] = model
    return model
print('Save/load helpers defined.')

### B3. Metrics, low-light adjustment and the prediction / "Not Yakshagana" logic
**How an image is judged (v3).**
1. The model outputs one probability per class: every Vesha type **plus** `NotYakshagana`.
2. If `P(NotYakshagana) >= OTHER_THRESHOLD` (default 0.50) → the answer is **"not related to Yakshagana"**. The model cannot say *what* the image is (Bharatanatya, a crowd, a selfie...), only that it does not look like Yakshagana.
3. Otherwise the image is treated as Yakshagana and the Vesha probabilities are **re-normalised among the Vesha types only**, so the confidence you see is "which Vesha, given that it is Yakshagana".

**Uncertainty.** Softmax confidence is *not* a guarantee of correctness. A Yakshagana prediction is flagged **uncertain** when the top probability is low, the top-2 margin is small, the distribution is flat (high entropy), `P(NotYakshagana)` is in the grey zone (`OTHER_WARN` to `OTHER_THRESHOLD`), or the image is tiny/blank. The thresholds are heuristics set in the config cell.


In [ ]:
def head_metrics(y_true, prob):
    y_pred = prob.argmax(axis=1)
    return dict(
        acc=accuracy_score(y_true, y_pred),
        precision=precision_score(y_true, y_pred, average='macro', zero_division=0),
        recall=recall_score(y_true, y_pred, average='macro', zero_division=0),
        f1=f1_score(y_true, y_pred, average='macro', zero_division=0),
    ), y_pred

# ---------- low-light simulation ----------
def adjust_image(img, brightness=1.0, contrast=1.0):
    """PIL version: brightness < 1 darkens, contrast < 1 flattens."""
    out = ImageEnhance.Brightness(img).enhance(brightness)
    return ImageEnhance.Contrast(out).enhance(contrast)

def adjust_tensor(x, brightness=1.0, contrast=1.0):
    """Tensor version for batches of 0-255 images (per-image contrast pivot = image mean)."""
    x = x * brightness
    m = tf.reduce_mean(x, axis=[-3, -2, -1], keepdims=True)
    return tf.clip_by_value((x - m) * contrast + m, 0.0, 255.0)

# ---------- single-image inference ----------
def load_rgb_image(source):
    img = ImageOps.exif_transpose(Image.open(source))
    return img.convert('RGB')

def pil_to_batch(img, img_size):
    arr = np.asarray(img, dtype=np.float32)
    return tf.image.resize(arr, (img_size, img_size)).numpy()[None, ...]    # same resize as training

def _other_index():
    other = ARTIFACTS.get('other_class')
    names = ARTIFACTS['classes']
    return names.index(other) if other in names else None

def interpret_probs(p, img=None):
    """Turn one probability vector into a decision: Not Yakshagana, or a Vesha type with an uncertainty check."""
    names = ARTIFACTS['classes']
    p = np.asarray(p, dtype=np.float64)
    oi = _other_index()
    p_other = float(p[oi]) if oi is not None else 0.0
    img_notes = []
    if img is not None:
        if min(img.size) < 64:
            img_notes.append('the image is very small')
        if np.asarray(img.convert('L'), dtype=np.float32).std() < 5:
            img_notes.append('the image is almost uniform (little visible content)')

    # ---- case 1: not Yakshagana ----
    if oi is not None and p_other >= cfg.OTHER_THRESHOLD:
        order = np.argsort(-p)
        return dict(is_yakshagana=False, label='Not Yakshagana', confidence=p_other, p_yak=1.0 - p_other,
                    top=[(names[i], float(p[i])) for i in order[:3]], probs=p,
                    uncertain=bool(img_notes), reasons=img_notes, open_set=True)

    # ---- case 2: Yakshagana -> which Vesha? (re-normalise over Vesha classes only) ----
    vi = [i for i in range(len(names)) if i != oi]
    pv = p[vi]
    pv = pv / max(pv.sum(), 1e-12)
    order = np.argsort(-pv)
    p1 = float(pv[order[0]]); p2 = float(pv[order[1]]) if len(pv) > 1 else 0.0
    ent = float(-(pv * np.log(pv + 1e-12)).sum() / np.log(len(pv))) if len(pv) > 1 else 0.0
    reasons = []
    if p1 < cfg.CONF_THRESHOLD:
        reasons.append(f'top Vesha confidence {p1:.0%} is below the {cfg.CONF_THRESHOLD:.0%} threshold')
    if len(pv) > 1 and (p1 - p2) < cfg.MARGIN_THRESHOLD:
        reasons.append(f'the top two Vesha types are close (margin {(p1 - p2):.0%})')
    if ent > cfg.ENTROPY_THRESHOLD:
        reasons.append(f'Vesha probabilities are spread out (normalised entropy {ent:.2f})')
    if oi is not None and p_other >= cfg.OTHER_WARN:
        reasons.append(f'there is a {p_other:.0%} chance this is NOT Yakshagana (grey zone)')
    reasons += img_notes
    return dict(is_yakshagana=True, label=names[vi[order[0]]], confidence=p1, p_yak=1.0 - p_other,
                top=[(names[vi[i]], float(pv[i])) for i in order[:3]], probs=p,
                uncertain=len(reasons) > 0, reasons=reasons, open_set=oi is not None)

def _pick_model(model_name=None):
    name = model_name or cfg.PREDICTION_MODEL or ARTIFACTS.get('best_model')
    if name is None:
        raise RuntimeError('No model selected. Run Part D first, or set cfg.PREDICTION_MODEL.')
    return name, get_model(name)

def predict_image(img, model_name=None, brightness=1.0, contrast=1.0):
    """Predict one PIL image: 'Not Yakshagana', or the Vesha type plus a trust judgement."""
    name, model = _pick_model(model_name)
    if brightness != 1.0 or contrast != 1.0:
        img = adjust_image(img, brightness, contrast)
    p = np.asarray(model.predict(pil_to_batch(img, ARTIFACTS['img_size']), verbose=0))[0]
    res = interpret_probs(p, img)
    res['model'] = name
    return res

def predict_batch(imgs, model_name=None, batch_size=32):
    """Predict many PIL images (handy for evaluating a folder of photos)."""
    name, model = _pick_model(model_name)
    x = np.concatenate([pil_to_batch(im, ARTIFACTS['img_size']) for im in imgs])
    P = np.asarray(model.predict(x, batch_size=batch_size, verbose=0))
    out = []
    for im, p in zip(imgs, P):
        r = interpret_probs(p, im); r['model'] = name; out.append(r)
    return out

def show_prediction(res, img, title=None):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw={'width_ratios': [1.2, 1]})
    axes[0].imshow(img); axes[0].axis('off'); axes[0].set_title(title or 'Uploaded image')
    top = res['top']
    axes[1].barh([n for n, _ in top][::-1], [p for _, p in top][::-1],
                 color='tab:red' if not res['is_yakshagana'] else 'tab:blue')
    axes[1].set_xlim(0, 1); axes[1].set_xlabel('probability')
    axes[1].set_title('Top predictions' if not res['is_yakshagana'] else 'Vesha type (given that it is Yakshagana)')
    plt.tight_layout(); plt.show()
    print(f'Model used      : {res["model"]}')
    if not res['is_yakshagana']:
        print(f'\n❌ NOT RELATED TO YAKSHAGANA  (P(not Yakshagana) = {res["confidence"]:.1%})')
        print('   This photo does not look like a Yakshagana performance. The model cannot tell you what it shows')
        print('   (e.g. Bharatanatya, a crowd, a landscape); it only knows it is not Yakshagana.')
        if res['reasons']:
            print('   Note:', '; '.join(res['reasons']))
        return
    top = res['top']
    print(f'P(Yakshagana)   : {res["p_yak"]:.1%}')
    print(f'Predicted Vesha : {top[0][0]}  (confidence {top[0][1]:.1%} among Vesha types)')
    print('Top predictions : ' + ' | '.join(f'{n} {p:.1%}' for n, p in top))
    if not res['open_set']:
        print('\nℹ️ This model was trained WITHOUT a NotYakshagana class, so it cannot reject non-Yakshagana photos. '
              'Add a NotYakshagana folder and retrain.')
    if res['uncertain']:
        print('\n⚠️ UNCERTAIN PREDICTION - please do not rely on it. Reasons:')
        for r in res['reasons']:
            print('   •', r)
        print('   The image may show a Vesha outside the trained classes, an unusual pose or lighting, '
              'or may not be a Yakshagana performer at all.')
    else:
        print('\n✅ Relatively confident, but the model only knows the Vesha types it was trained on, and '
              'confidence scores are not guarantees. Treat this as a suggestion, not a fact.')
print('Inference helpers defined.')

## Part C — Dataset: load, validate, clean, split

### C1. Get the dataset into the notebook
Choose `DATA_SOURCE` in the config cell:
* `'zip'`: a file picker appears; select a ZIP of your dataset folder (for very large ZIPs, put it on Drive and use `'drive'`).
* `'drive'`: reads a folder or `.zip` under `My Drive/<DRIVE_DATA_SUBPATH>`.
* `'local'`: uses a folder already on the Colab disk.

A single wrapper folder inside the ZIP (e.g. `dataset/`) is skipped automatically. To override, set `DATASET_ROOT_OVERRIDE`.

In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp', '.tif', '.tiff'}

def extract_zip(zip_path, dest):
    if os.path.exists(dest):
        shutil.rmtree(dest)
    os.makedirs(dest)
    with zipfile.ZipFile(zip_path) as zf:
        bad = zf.testzip()
        if bad:
            print(f'⚠️ ZIP reports a damaged member: {bad} (continuing)')
        zf.extractall(dest)

def find_dataset_root(path):
    """Descend through single wrapper folders until we reach the folder that holds the Vesha folders."""
    if cfg.DATASET_ROOT_OVERRIDE:
        return Path(cfg.DATASET_ROOT_OVERRIDE)
    path = Path(path)
    while True:
        subdirs = [d for d in path.iterdir() if d.is_dir() and not d.name.startswith(('.', '__'))]
        imgs = [f for f in path.iterdir() if f.is_file() and f.suffix.lower() in IMG_EXTS]
        if len(subdirs) == 1 and not imgs:
            path = subdirs[0]
        else:
            return path

def prepare_dataset():
    src = cfg.DATA_SOURCE
    work_ds = os.path.join(cfg.WORK_DIR, 'dataset')
    if src == 'zip':
        if not IN_COLAB:
            raise RuntimeError("DATA_SOURCE='zip' needs Google Colab's upload widget. Use 'local' or 'drive'.")
        from google.colab import files
        print('Select your dataset .zip file...')
        uploaded = files.upload()
        zips = [n for n in uploaded if n.lower().endswith('.zip')]
        if not zips:
            raise FileNotFoundError('No .zip file was uploaded.')
        extract_zip(os.path.abspath(zips[0]), work_ds)
        os.remove(os.path.abspath(zips[0]))
        root = work_ds
    elif src == 'drive':
        if not DRIVE_MOUNTED:
            raise RuntimeError('Google Drive is not mounted. Re-run cell A5.')
        p = os.path.join(DRIVE_ROOT, cfg.DRIVE_DATA_SUBPATH)
        if not os.path.exists(p):
            raise FileNotFoundError(f'Not found on Drive: {p}. Check DRIVE_DATA_SUBPATH.')
        if p.lower().endswith('.zip'):
            extract_zip(p, work_ds); root = work_ds
        elif cfg.COPY_DRIVE_FOLDER_TO_LOCAL:
            print('Copying dataset from Drive to the Colab disk (faster training)...')
            shutil.copytree(p, work_ds, dirs_exist_ok=True); root = work_ds
        else:
            root = p
    else:
        if not os.path.isdir(cfg.LOCAL_DATA_PATH):
            raise FileNotFoundError(f'Folder not found: {cfg.LOCAL_DATA_PATH}')
        root = cfg.LOCAL_DATA_PATH
    return find_dataset_root(root)

try:
    DATASET_ROOT = prepare_dataset()
except Exception as e:
    print('❌ Could not prepare the dataset:', e)
    raise
print('Dataset root:', DATASET_ROOT)
print('Top-level folders (expected: Vesha types):', sorted(d.name for d in Path(DATASET_ROOT).iterdir() if d.is_dir()))

### C2. Scan the folder structure and find missing labels
The label is the **top-level folder name**. Any deeper character folder is ignored.

**Missing labels.** An image has no label if it is not inside a Vesha folder (e.g. `img.jpg` sitting at the top level). The notebook never guesses a label: such files are **excluded** and listed in a table. Fix by moving each into the correct Vesha folder and re-running; if they are truly unlabeled, have someone knowledgeable label them first. A wrong label does more harm than a missing image.

In [ ]:
root = Path(DATASET_ROOT)
rows, ignored_ext = [], Counter()
for p in sorted(root.rglob('*')):
    if not p.is_file():
        continue
    rel = p.relative_to(root)
    if any(part.startswith('.') or part == '__MACOSX' for part in rel.parts):
        continue
    if p.suffix.lower() not in IMG_EXTS:
        ignored_ext[p.suffix.lower() or '(no extension)'] += 1
        continue
    parts = rel.parts
    rows.append(dict(
        filepath=str(p), relpath=rel.as_posix(),
        vesha=parts[0].strip() if len(parts) >= 2 else None,
        # Vesha/Character/Performer/img -> the performer folder becomes a group (the character folder is ignored)
        explicit_group=('/'.join(parts[:-1]) if len(parts) >= 4 else None),
    ))
df_raw = pd.DataFrame(rows)
if df_raw.empty:
    raise FileNotFoundError('No images were found. Check the dataset path and folder structure.')
if cfg.GROUP_CSV:
    gcsv = pd.read_csv(cfg.GROUP_CSV)
    gmap = dict(zip(gcsv['relpath'], gcsv['group'].astype(str)))
    df_raw['explicit_group'] = df_raw['relpath'].map(lambda r: ('csv:' + gmap[r]) if r in gmap else None)
    print(f'Loaded group labels for {df_raw.explicit_group.notna().sum()} images from {cfg.GROUP_CSV}')

df_raw['idx'] = np.arange(len(df_raw))
print(f'Found {len(df_raw)} image files.')
if ignored_ext:
    print('Ignored non-image files:', dict(ignored_ext))

missing = df_raw[df_raw.vesha.isna()]
if len(missing):
    print(f'\n⚠️ {len(missing)} image(s) have MISSING LABELS (not inside a Vesha folder) and will be excluded:')
    display(missing[['relpath']].head(20))
    missing[['relpath']].to_csv(os.path.join(cfg.WORK_DIR, 'missing_labels.csv'), index=False)
else:
    print('✅ Every image sits inside a Vesha folder.')
df = df_raw.drop(missing.index).reset_index(drop=True)

### C3. Clean the label names and look at the classes
Folders named like the "not Yakshagana" class (`NotYakshagana`, `Other`, `NonYakshagana`, `Unknown`... see `OTHER_ALIASES`) are all merged into **one** class called `cfg.OTHER_CLASS_NAME`. Other spelling variants such as `Rajavesha`, `rajavesha` and `Raja_vesha` are merged into the most common spelling. Review the printed merges; if two *different* classes were merged by mistake, rename the folders.


In [ ]:
OTHER = cfg.OTHER_CLASS_NAME
_norm = lambda s: re.sub(r'[\s_\-]+', '', str(s).lower())
_alias_keys = {_norm(a) for a in list(cfg.OTHER_ALIASES) + [OTHER]}
is_other = df['vesha'].map(_norm).isin(_alias_keys)
df.loc[is_other, 'vesha'] = OTHER                 # every alias becomes the one "not Yakshagana" class

def canonicalize_labels(series, kind):
    key = series.str.lower().str.replace(r'[\s_\-]+', '', regex=True)
    mapping = {}
    for k in key.unique():
        variants = series[key == k]
        best = variants.value_counts().idxmax()
        for v in variants.unique():
            mapping[v] = best
        if variants.nunique() > 1:
            print(f'  merged {kind} spellings {sorted(variants.unique())} -> "{best}"')
    return series.map(mapping)

df['vesha'] = canonicalize_labels(df['vesha'], 'Vesha')
print(f'"Not Yakshagana" images found (folder -> "{OTHER}"): {int((df.vesha == OTHER).sum())}')
if (df.vesha == OTHER).sum() == 0:
    print('⚠️ No NotYakshagana folder found. The model will still train, but it will NOT be able to reject '
          'non-Yakshagana photos. Add dataset/NotYakshagana/ with Bharatanatya, crowds, etc. and re-run from C1.')
print('\nImages per class:')
print(df['vesha'].value_counts().to_string())

### C4. Validate every image and build a clean cache
Each image is opened and fully verified. For every valid image we also compute an MD5 fingerprint (exact duplicates) and a perceptual hash (near-duplicates), and save a **clean RGB JPEG copy** (EXIF rotation fixed, longer side ≤ `MAX_CACHE_SIDE`). Training reads these local copies, which avoids odd formats (CMYK, WebP, TIFF) and is much faster than reading from Drive.

**Corrupted images** (unreadable, truncated, or smaller than `MIN_IMAGE_SIDE`) are dropped and listed. **Exact duplicates** are reduced to one copy. If the *same* file appears under two *different* Vesha labels, that is label noise, so **all copies are dropped** and reported.

In [ ]:
CACHE_DIR = os.path.join(cfg.WORK_DIR, 'clean_cache')
os.makedirs(CACHE_DIR, exist_ok=True)

def inspect_and_cache(args):
    idx, path = args
    out = dict(idx=idx, ok=False, error=None)
    try:
        with open(path, 'rb') as f:
            raw = f.read()
        md5 = hashlib.md5(raw).hexdigest()
        with Image.open(io.BytesIO(raw)) as im:
            im.verify()                                     # detects truncated / damaged files
        im = ImageOps.exif_transpose(Image.open(io.BytesIO(raw))).convert('RGB')
        w, h = im.size
        if min(w, h) < cfg.MIN_IMAGE_SIDE:
            raise ValueError(f'image too small ({w}x{h})')
        g = np.asarray(im.convert('L').resize((9, 8), Image.LANCZOS), dtype=np.int16)
        bits = (g[:, 1:] > g[:, :-1]).flatten()            # 64-bit difference hash (dHash)
        dhash = int(''.join('1' if b else '0' for b in bits), 2)
        scale = cfg.MAX_CACHE_SIDE / max(w, h)
        if scale < 1:
            im = im.resize((max(1, round(w * scale)), max(1, round(h * scale))), Image.LANCZOS)
        cache_path = os.path.join(CACHE_DIR, f'{idx:07d}.jpg')
        im.save(cache_path, quality=95)
        out.update(ok=True, md5=md5, dhash=dhash, cache_path=cache_path)
    except Exception as e:
        out['error'] = f'{type(e).__name__}: {e}'
    return out

with ThreadPoolExecutor(max_workers=8) as ex:
    results = list(tqdm(ex.map(inspect_and_cache, zip(df['idx'], df['filepath'])), total=len(df), desc='Validating images'))
df = df.merge(pd.DataFrame(results), on='idx', how='left')

bad = df[~df['ok'].fillna(False).astype(bool)]
if len(bad):
    print(f'\n⚠️ {len(bad)} corrupted/unreadable image(s) dropped:')
    display(bad[['relpath', 'error']].head(20))
    bad[['relpath', 'error']].to_csv(os.path.join(cfg.WORK_DIR, 'corrupted_images.csv'), index=False)
else:
    print('✅ No corrupted images found.')
df = df[df['ok'].fillna(False).astype(bool)].drop(columns=['ok', 'error']).reset_index(drop=True)

conflict_md5 = [m for m, g in df.groupby('md5') if len(g) > 1 and g['vesha'].nunique() > 1]
if conflict_md5:
    print(f'⚠️ {len(conflict_md5)} file(s) appear under DIFFERENT Vesha labels (label noise). All copies dropped:')
    display(df[df.md5.isin(conflict_md5)][['relpath', 'vesha']].head(20))
    df = df[~df.md5.isin(conflict_md5)]
n_dup = int(df.duplicated('md5').sum())
if n_dup:
    print(f'Found {n_dup} exact duplicate image(s).', 'Dropping them.' if cfg.DROP_EXACT_DUPLICATES else 'Keeping them (they share a group).')
    if cfg.DROP_EXACT_DUPLICATES:
        df = df.drop_duplicates('md5')
df = df.sort_values('relpath').reset_index(drop=True)
print(f'\nUsable images: {len(df)}')

### C5. Group images to prevent data leakage
Photos from one performance (burst shots, same performer, same pose) look almost identical. If one lands in training and its twin in testing, the test score becomes **unrealistically high**. Every image gets a **group id** and the splitter keeps each group entirely inside one subset. Groups merge three sources:
1. **performer / performance folders** (`Vesha/Character/PerformerA/…`) or a `GROUP_CSV` (best option),
2. **near-duplicate detection** with a perceptual hash,
3. exact duplicates (already removed).

⚠️ Near-duplicate hashing catches *visually similar frames* only. It cannot know that two different-looking photos show the same performer, so add performer folders or a `GROUP_CSV` if you can.

In [ ]:
class DSU:
    """Union-find: merges images that belong to the same group."""
    def __init__(self, n): self.p = list(range(n))
    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]; a = self.p[a]
        return a
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[rb] = ra

def popcount64(x):
    x = x - ((x >> np.uint64(1)) & np.uint64(0x5555555555555555))
    x = (x & np.uint64(0x3333333333333333)) + ((x >> np.uint64(2)) & np.uint64(0x3333333333333333))
    x = (x + (x >> np.uint64(4))) & np.uint64(0x0F0F0F0F0F0F0F0F)
    return ((x * np.uint64(0x0101010101010101)) >> np.uint64(56)).astype(np.int32)

n = len(df)
dsu = DSU(n)
first_of = {}
for i, g in enumerate(df['explicit_group'].values):          # 1) performer folders / CSV
    if isinstance(g, str):
        if g in first_of: dsu.union(first_of[g], i)
        else: first_of[g] = i

if n <= 30000:                                               # 2) near-duplicates
    hashes = np.array(df['dhash'].tolist(), dtype=np.uint64)
    for i in tqdm(range(n - 1), desc='Near-duplicate search'):
        d = popcount64(hashes[i] ^ hashes[i + 1:])
        for j in np.nonzero(d <= cfg.NEAR_DUP_HAMMING)[0]:
            dsu.union(i, i + 1 + int(j))
else:
    print('⚠️ Large dataset: skipping near-duplicate search. Provide performer folders or GROUP_CSV instead.')

df['group'] = [dsu.find(i) for i in range(n)]
gs = df.groupby('group').size()
print(f'{len(gs)} groups for {n} images | largest group: {gs.max()} images | '
      f'images in multi-image groups: {(df.group.map(gs) > 1).mean():.1%}')
if gs.max() > 0.2 * n:
    print('⚠️ One group holds >20% of all images; the split may be unbalanced. Lower NEAR_DUP_HAMMING or check group folders.')

mixed = df.groupby('group')['vesha'].nunique() > 1
if mixed.any():
    print(f'⚠️ {int(mixed.sum())} group(s) contain near-identical images with DIFFERENT Vesha labels. Please review:')
    display(df[df.group.isin(mixed[mixed].index)][['relpath', 'vesha']].head(20))

### C6. Drop very small classes, cap the "NotYakshagana" class, encode labels, inspect class balance
A class with fewer than `MIN_IMAGES_PER_CLASS` images cannot be split 70/15/15 reliably, so it is dropped (and reported). Collecting more images is the real fix.

**The NotYakshagana class** is capped at `MAX_OTHER_RATIO` x the average Vesha class size so that it cannot swamp the Vesha types. It is always placed **last** in the class list.

**Class imbalance.** What the notebook does about it:
* the training loss is **class-weighted** (`USE_CLASS_WEIGHTS`) so rare classes are not ignored;
* the split is **stratified**, so each class keeps about the same share in every subset;
* we report **macro-averaged** precision, recall and F1 (every class counts equally), because accuracy alone can look good while a small class is always wrong.


In [ ]:
OTHER = cfg.OTHER_CLASS_NAME
counts = df['vesha'].value_counts()
rare = counts[counts < cfg.MIN_IMAGES_PER_CLASS]
if len(rare):
    print(f'⚠️ Dropping {len(rare)} class(es) with fewer than {cfg.MIN_IMAGES_PER_CLASS} images:')
    print(rare.to_string())
    df = df[~df['vesha'].isin(rare.index)].reset_index(drop=True)

yak_names = sorted(c for c in df['vesha'].unique() if c != OTHER)
if len(yak_names) < 1:
    raise ValueError('Need at least 1 Vesha type after cleaning. Check the dataset structure.')

has_other = bool((df['vesha'] == OTHER).any())
if has_other:
    mean_size = df[df.vesha != OTHER]['vesha'].value_counts().mean()
    cap = int(cfg.MAX_OTHER_RATIO * mean_size)
    n_other = int((df.vesha == OTHER).sum())
    if n_other > cap:
        keep_idx = df[df.vesha == OTHER].sample(cap, random_state=SEED).index
        df = df[(df.vesha != OTHER) | df.index.isin(keep_idx)].reset_index(drop=True)
        print(f'ℹ️ NotYakshagana capped from {n_other} to {cap} images (MAX_OTHER_RATIO={cfg.MAX_OTHER_RATIO} x average Vesha size {mean_size:.0f}).')
else:
    print('⚠️ No (usable) NotYakshagana class: the model will not be able to reject non-Yakshagana photos.')

vesha_classes = yak_names + ([OTHER] if has_other else [])      # model output order; NotYakshagana is last
if len(vesha_classes) < 2:
    raise ValueError('Need at least 2 classes (e.g. 1 Vesha + NotYakshagana, or 2 Vesha types).')
df['vesha_id'] = df['vesha'].map({c: i for i, c in enumerate(vesha_classes)}).astype(int)

ARTIFACTS.clear()
ARTIFACTS.update(classes=vesha_classes, other_class=(OTHER if has_other else None),
                 img_size=cfg.IMG_SIZE, models=[], best_model=None)
save_artifacts()

vc = df['vesha'].value_counts()
print(f'\n{len(df)} images | {len(vesha_classes)} classes: {vesha_classes}')
print(f'Class imbalance (largest/smallest class): {vc.max() / vc.min():.1f}x')
if vc.max() / vc.min() > 3:
    print('⚠️ Notable imbalance: rely on macro-F1, not accuracy alone.')

plt.figure(figsize=(8, max(3, 0.6 * len(vc))))
vc.sort_values().plot.barh(color=['tab:red' if c == OTHER else 'tab:blue' for c in vc.sort_values().index])
plt.title('Images per class (red = NotYakshagana)'); plt.xlabel('images'); plt.show()

fig, axes = plt.subplots(len(vesha_classes), 4, figsize=(12, 3 * len(vesha_classes)), squeeze=False)
for r, v in enumerate(vesha_classes):
    sample = df[df.vesha == v].sample(min(4, (df.vesha == v).sum()), random_state=SEED)
    for c in range(4):
        axes[r, c].axis('off')
        if c < len(sample):
            axes[r, c].imshow(Image.open(sample.iloc[c].cache_path)); axes[r, c].set_title(v, fontsize=9)
plt.tight_layout(); plt.show()

### C7. Stratified, group-aware 70 / 15 / 15 split
* **Stratified** on the Vesha label, so each class keeps roughly the same proportion in train, validation and test.
* **Group-aware**: all images of a group go to the **same** subset, so near-twins cannot leak between subsets.
* The data are cut into 20 folds of ~5%; 14 become train, 3 validation, 3 test. Several random seeds are tried, and the split where every class appears in all three subsets (sizes closest to 70/15/15) is kept. It is reproducible.
* **The same split is used for every model**, so the comparison is fair.
* **The test set is not used for training, early stopping, learning-rate scheduling or model selection.** It is evaluated once, in Part D4.

⚠️ With very small classes an exact 70/15/15 split per class is impossible; check the counts below.

In [ ]:
def assign_split(frame, seed, n_folds, n_test, n_val):
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    fold = np.full(len(frame), -1)
    for k, (_, idx) in enumerate(sgkf.split(frame, frame['vesha'], groups=frame['group'])):
        fold[idx] = k
    perm = np.random.RandomState(seed).permutation(n_folds)
    test_f, val_f = perm[:n_test], perm[n_test:n_test + n_val]
    return np.where(np.isin(fold, test_f), 'test', np.where(np.isin(fold, val_f), 'val', 'train'))

n_groups = df['group'].nunique()
n_folds = min(cfg.N_FOLDS, n_groups)
n_test = max(1, round(n_folds * cfg.SPLIT[2]))
n_val = max(1, round(n_folds * cfg.SPLIT[1]))
if n_test + n_val >= n_folds:
    raise ValueError(f'Only {n_groups} groups: too few for a train/val/test split. Add more images or performer information.')
if n_folds < 20:
    print(f'⚠️ Only {n_folds} folds possible (few groups), so split sizes will be approximate.')

best, last_err = None, None
for attempt in range(cfg.SPLIT_ATTEMPTS):
    seed_try = cfg.SEED + attempt
    try:
        split = assign_split(df, seed_try, n_folds, n_test, n_val)
    except ValueError as e:
        last_err = e; continue
    ct = pd.crosstab(df['vesha'], split).reindex(columns=['train', 'val', 'test'], fill_value=0)
    coverage = int((ct > 0).all(axis=1).sum())
    sizes = pd.Series(split).value_counts(normalize=True).reindex(['train', 'val', 'test']).fillna(0)
    size_err = abs(sizes['val'] - cfg.SPLIT[1]) + abs(sizes['test'] - cfg.SPLIT[2])
    key = (coverage, -size_err)
    if best is None or key > best[0]:
        best = (key, split, seed_try)
    if coverage == df['vesha'].nunique() and size_err < 0.03:
        break
if best is None:
    raise RuntimeError(f'Could not build a split: {last_err}')

df['split'] = best[1]
print(f'Split seed used: {best[2]}')
train_df = df[df.split == 'train'].reset_index(drop=True)
val_df = df[df.split == 'val'].reset_index(drop=True)
test_df = df[df.split == 'test'].reset_index(drop=True)
print(f'Train {len(train_df)} ({len(train_df)/len(df):.1%}) | Val {len(val_df)} ({len(val_df)/len(df):.1%}) | Test {len(test_df)} ({len(test_df)/len(df):.1%})')

order = ['train', 'val', 'test']
tbl = pd.crosstab(df['vesha'], df['split']).reindex(columns=order, fill_value=0)
tbl['total'] = tbl.sum(axis=1)
print('\nImages per Vesha class in each split:')
display(tbl)

sets = {s: df[df.split == s] for s in order}
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    assert not set(sets[a].relpath) & set(sets[b].relpath), f'Same file in {a} and {b}!'
    assert not set(sets[a].md5) & set(sets[b].md5), f'Identical image content in {a} and {b}!'
    assert not set(sets[a].group) & set(sets[b].group), f'Same group in {a} and {b}!'
print('✅ Verified: no image (file path or identical content) and no group appears in more than one subset.')
for s in order:
    miss = sorted(set(df.vesha) - set(sets[s].vesha))
    if miss:
        print(f'⚠️ These classes have NO images in {s}: {miss}. Metrics for them are undefined; add more data/groups.')

df[['relpath', 'vesha', 'group', 'split']].to_csv(os.path.join(SAVE_DIR, 'split.csv'), index=False)
print('Split saved to', os.path.join(SAVE_DIR, 'split.csv'))

### C8. Build the input pipelines
* **Training**: shuffled, with optional per-sample class weights computed **from the training split only**. Augmentation happens inside the model and is active only during training.
* **Validation / test**: no shuffling, no weights, **no augmentation**.

The preview shows augmentation on a few **training** images only.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image_tf(path):
    raw = tf.io.read_file(path)
    img = tf.io.decode_jpeg(raw, channels=3)                       # cache files are always JPEG
    return tf.image.resize(tf.cast(img, tf.float32), (cfg.IMG_SIZE, cfg.IMG_SIZE))   # pixel range stays 0-255

_present = np.unique(train_df['vesha_id'].values)
class_w = np.ones(len(vesha_classes), dtype=np.float32)
class_w[_present] = compute_class_weight('balanced', classes=_present, y=train_df['vesha_id'].values)
print('Class weights (training split):', {vesha_classes[i]: round(float(w), 2) for i, w in enumerate(class_w)})

def make_dataset(frame, training):
    paths = frame['cache_path'].astype(str).values
    y = frame['vesha_id'].values.astype(np.int32)
    if training and cfg.USE_CLASS_WEIGHTS:
        ds = tf.data.Dataset.from_tensor_slices((paths, y, class_w[y].astype(np.float32)))
        ds = ds.shuffle(len(frame), seed=cfg.SEED, reshuffle_each_iteration=True)
        ds = ds.map(lambda p, l, w: (load_image_tf(p), l, w), num_parallel_calls=AUTOTUNE)
    else:
        ds = tf.data.Dataset.from_tensor_slices((paths, y))
        if training:
            ds = ds.shuffle(len(frame), seed=cfg.SEED, reshuffle_each_iteration=True)
        ds = ds.map(lambda p, l: (load_image_tf(p), l), num_parallel_calls=AUTOTUNE)
    return ds.batch(cfg.BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_eval_ds = make_dataset(test_df, training=False)    # built now, NOT used until the final evaluation
print(f'Batches: train {len(train_ds)}, val {len(val_ds)}, test {len(test_eval_ds)} (batch size {cfg.BATCH_SIZE})')

prev = train_df.sample(min(6, len(train_df)), random_state=SEED)
x_prev, _ = next(iter(make_dataset(prev, training=False)))
x_aug = build_augmentation()(x_prev, training=True).numpy()
fig, axes = plt.subplots(2, len(prev), figsize=(3 * len(prev), 6), squeeze=False)
for i in range(len(prev)):
    axes[0, i].imshow(np.clip(x_prev[i].numpy(), 0, 255).astype('uint8')); axes[0, i].axis('off'); axes[0, i].set_title('original')
    axes[1, i].imshow(np.clip(x_aug[i], 0, 255).astype('uint8')); axes[1, i].axis('off'); axes[1, i].set_title('augmented (train only)')
plt.tight_layout(); plt.show()

## Part D — Train, compare and evaluate

### D1. Train all models
Each model uses the same two-phase recipe on the **same** training and validation data:
1. **Head training** (`EPOCHS_HEAD`, backbone frozen, `LR_HEAD`),
2. **Fine-tuning** (`EPOCHS_FINE`, last `FINE_TUNE_LAYERS` backbone layers unfrozen, small `LR_FINE`).

In both phases **early stopping** watches validation loss (and restores the best weights) and **ReduceLROnPlateau** lowers the learning rate when validation loss stalls. The test set is never touched here.

Weights and a small history file are saved to Drive after each model, so if Colab disconnects you can re-run this cell and finished models are skipped (`FORCE_RETRAIN = False`). "Training time" includes the per-epoch validation passes.

In [ ]:
def train_one_model(name):
    set_global_seed(cfg.SEED)
    MODEL_CACHE.clear()
    keras.backend.clear_session()
    try:
        model, base = build_model(name, len(vesha_classes), cfg.IMG_SIZE, pretrained=True, dropout=cfg.DROPOUT)
    except Exception as e:
        raise RuntimeError(f'Could not build {name} with ImageNet weights (is the internet available?): {e}')

    t0 = time.time()
    compile_model(model, cfg.LR_HEAD)
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=cfg.EPOCHS_HEAD, callbacks=make_callbacks(), verbose=2)
    hist = {k: [float(x) for x in v] for k, v in h1.history.items()}
    n1, n2 = len(h1.history['loss']), 0
    if cfg.EPOCHS_FINE > 0:
        set_fine_tuning(base, cfg.FINE_TUNE_LAYERS)
        compile_model(model, cfg.LR_FINE)                    # recompile after changing trainable layers
        h2 = model.fit(train_ds, validation_data=val_ds, epochs=cfg.EPOCHS_FINE, callbacks=make_callbacks(), verbose=2)
        n2 = len(h2.history['loss'])
        for k, v in h2.history.items():
            hist.setdefault(k, []).extend(float(x) for x in v)
    train_time = time.time() - t0

    model.save_weights(model_weights_path(name))
    meta = dict(model=name, train_time_sec=train_time, epochs_phase1=n1, epochs_phase2=n2, history=hist,
                batch_size=cfg.BATCH_SIZE, img_size=cfg.IMG_SIZE, seed=cfg.SEED)
    with open(meta_path(name), 'w') as f:
        json.dump(meta, f)
    return meta

trained = []
for name in cfg.MODELS_TO_TRAIN:
    if os.path.exists(model_weights_path(name)) and os.path.exists(meta_path(name)) and not cfg.FORCE_RETRAIN:
        print(f'⏭️ {name}: saved weights found, skipping (set FORCE_RETRAIN=True to retrain).')
        trained.append(name); continue
    print(f'\n{"=" * 70}\nTraining {name}\n{"=" * 70}')
    try:
        meta = train_one_model(name)
        print(f'✅ {name} finished in {meta["train_time_sec"] / 60:.1f} min '
              f'({meta["epochs_phase1"]} head epochs + {meta["epochs_phase2"]} fine-tune epochs)')
        trained.append(name)
    except tf.errors.ResourceExhaustedError:
        print(f'❌ {name}: GPU out of memory. Lower BATCH_SIZE (e.g. 16) and re-run this cell.')
    except Exception as e:
        print(f'❌ {name} failed: {e}')

if not trained:
    raise RuntimeError('No model was trained successfully. See the messages above.')
ARTIFACTS['models'] = trained
save_artifacts()
print('\nAvailable models:', trained)

### D1b. Backup the trained models
Right after training, this zips the weights + class mapping and downloads them. Keep this zip as a safety copy in case Drive is unavailable.

In [ ]:
# BACKUP: zip the saved models and download them to your computer (safety net)
import shutil
zip_base = os.path.join(cfg.WORK_DIR, 'yakshagana_vesha_models')
shutil.make_archive(zip_base, 'zip', SAVE_DIR)
print('Models in', SAVE_DIR, ':', sorted(os.listdir(SAVE_DIR)))
print('Backup zip:', zip_base + '.zip', f'({os.path.getsize(zip_base + ".zip") / 1e6:.0f} MB)')
if IN_COLAB:
    from google.colab import files
    files.download(zip_base + '.zip')


### D2. Training and validation curves
Loss and accuracy per epoch for every model. The dashed line marks the start of fine-tuning. A growing gap between training and validation curves indicates **overfitting**, common with small datasets.

In [ ]:
METAS = {}
for n_ in trained:
    with open(meta_path(n_)) as f:
        METAS[n_] = json.load(f)

fig, axes = plt.subplots(len(METAS), 2, figsize=(13, 3.6 * len(METAS)), squeeze=False)
for r, (name, m) in enumerate(METAS.items()):
    h = m['history']
    for c, (title, key) in enumerate([('Loss', 'loss'), ('Accuracy', 'accuracy')]):
        ax = axes[r, c]
        ax.plot(range(1, len(h[key]) + 1), h[key], label='train')
        ax.plot(range(1, len(h['val_' + key]) + 1), h['val_' + key], label='validation')
        if m['epochs_phase2'] > 0:
            ax.axvline(m['epochs_phase1'] + 0.5, color='gray', ls='--', lw=1)
        ax.set_title(f'{name}: {title}'); ax.set_xlabel('epoch'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 4.5))
for name, m in METAS.items():
    plt.plot(range(1, len(m['history']['val_accuracy']) + 1), m['history']['val_accuracy'], label=name)
plt.title('Validation accuracy, all models'); plt.xlabel('epoch'); plt.legend(); plt.grid(alpha=.3); plt.show()

### D3. Choose the best model using the VALIDATION set only
Choosing the winner by test score would quietly turn the test set into a second validation set and make the final number optimistic. The best model is therefore selected here by **validation macro-F1**. The test set is evaluated afterwards, once, for reporting.

In [ ]:
VAL_RESULTS = {}
y_val = val_df['vesha_id'].values
for name in trained:
    p = np.asarray(get_model(name).predict(val_ds, verbose=0))
    m, _ = head_metrics(y_val, p)
    VAL_RESULTS[name] = dict(val_acc=m['acc'], val_macro_f1=m['f1'])
val_table = pd.DataFrame(VAL_RESULTS).T.sort_values('val_macro_f1', ascending=False)
display(val_table.round(4))

BEST_MODEL = val_table.index[0]
ARTIFACTS['best_model'] = BEST_MODEL
save_artifacts()
print(f'🏆 Best model on the validation set: {BEST_MODEL}  (saved to class_mapping.json)')
if len(val_table) > 1 and abs(val_table.val_macro_f1.iloc[0] - val_table.val_macro_f1.iloc[1]) < 0.01:
    print('ℹ️ The top two models are within 1 point on validation. With a small validation set that may be noise.')

### D4. FINAL evaluation on the untouched test set
First use of the test set. For **every** model: accuracy, macro precision, macro recall, macro F1, training time and inference time. Inference time is measured on already-loaded images (no disk reading): *batch* = ms per image when predicting in batches, *single* = latency for one image at a time.

In [ ]:
def time_inference(model, bench_x, repeats=3):
    model.predict(bench_x[:cfg.BATCH_SIZE], verbose=0)                  # warm-up
    t = []
    for _ in range(repeats):
        s = time.perf_counter(); model.predict(bench_x, batch_size=cfg.BATCH_SIZE, verbose=0); t.append(time.perf_counter() - s)
    batch_ms = 1000 * float(np.median(t)) / len(bench_x)
    n_single = min(20, len(bench_x)); _ = np.asarray(model(bench_x[:1], training=False))
    s = time.perf_counter()
    for i in range(n_single):
        _ = np.asarray(model(bench_x[i:i + 1], training=False))
    return batch_ms, 1000 * (time.perf_counter() - s) / n_single

bench_x = np.concatenate([x.numpy() for x, _ in test_eval_ds.take(4)])
y_test = test_df['vesha_id'].values
TEST, rows = {}, []
for name in trained:
    model = get_model(name)
    p = np.asarray(model.predict(test_eval_ds, verbose=0))
    m, y_hat = head_metrics(y_test, p)
    b_ms, s_ms = time_inference(model, bench_x)
    TEST[name] = dict(prob=p, pred=y_hat)
    rows.append({'Model': name, 'Params (M)': model.count_params() / 1e6,
                 'Train time (min)': METAS[name]['train_time_sec'] / 60,
                 'Epochs run': METAS[name]['epochs_phase1'] + METAS[name]['epochs_phase2'],
                 'Accuracy': m['acc'], 'Macro Precision': m['precision'], 'Macro Recall': m['recall'], 'Macro F1': m['f1'],
                 'Infer ms/img (batch)': b_ms, 'Infer ms/img (single)': s_ms})
comparison = pd.DataFrame(rows).set_index('Model')
comparison.to_csv(os.path.join(SAVE_DIR, 'test_comparison.csv'))

print('Test-set results (macro = every Vesha class counts equally):')
try:
    display(comparison.style.format('{:.4f}')
            .highlight_max(subset=['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1'], color='#c8f7c5')
            .highlight_min(subset=['Train time (min)', 'Infer ms/img (batch)', 'Infer ms/img (single)'], color='#c8f7c5'))
except Exception:
    display(comparison.round(4))

best_test = comparison['Macro F1'].idxmax()
print(f'\nBest by TEST macro-F1: {best_test} | Chosen on VALIDATION (used for prediction): {BEST_MODEL}')
if best_test != BEST_MODEL:
    print('ℹ️ They differ. The validation-based choice is the statistically clean one; the gap is probably within noise on small data.')

### D5. Comparison charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
comparison[['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1']].plot.bar(ax=axes[0], rot=15)
axes[0].set_title('Test performance'); axes[0].set_ylim(0, 1); axes[0].grid(alpha=.3)
comparison['Train time (min)'].plot.bar(ax=axes[1], color='tab:purple', rot=15)
axes[1].set_title('Training time (minutes)'); axes[1].grid(alpha=.3)
comparison[['Infer ms/img (batch)', 'Infer ms/img (single)']].plot.bar(ax=axes[2], rot=15)
axes[2].set_title('Inference time (ms per image)'); axes[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 5))
for name in comparison.index:
    r = comparison.loc[name]
    plt.scatter(r['Infer ms/img (batch)'], r['Macro F1'], s=120, label=name)
plt.xlabel('Inference time (ms/img, batch)'); plt.ylabel('Macro F1 (test)')
plt.title('Accuracy vs. speed trade-off'); plt.legend(); plt.grid(alpha=.3); plt.show()

### D6. Confusion matrices and per-class reports
Rows = true Vesha, columns = predicted Vesha (normalised per row, so the diagonal is per-class recall). Look for Vesha pairs that get confused. Classes with few test images give unreliable per-class numbers.

In [ ]:
def plot_cm(y_true, y_pred, names, title):
    labels = np.unique(np.concatenate([y_true, y_pred]))
    cm = confusion_matrix(y_true, y_pred, labels=labels).astype(float)
    cmn = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    size = max(4.5, 0.9 * len(labels) + 2)
    fig, axes = plt.subplots(1, 2, figsize=(2 * size, size * 0.85))
    for ax, data, fmt, sub in [(axes[0], cm, '.0f', 'counts'), (axes[1], cmn, '.2f', 'row-normalised')]:
        sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues', ax=ax,
                    xticklabels=[names[i] for i in labels], yticklabels=[names[i] for i in labels])
        ax.set_title(f'{title} ({sub})'); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        ax.tick_params(axis='x', rotation=30)
    plt.tight_layout(); plt.show()

os.makedirs(os.path.join(SAVE_DIR, 'reports'), exist_ok=True)
for name in trained:
    print('=' * 80, f'\n{name}\n' + '=' * 80)
    y_hat = TEST[name]['pred']
    plot_cm(y_test, y_hat, vesha_classes, f'{name}')
    labels = np.unique(np.concatenate([y_test, y_hat]))
    rep = classification_report(y_test, y_hat, labels=labels, target_names=[vesha_classes[i] for i in labels],
                                zero_division=0, digits=3)
    print(rep)
    with open(os.path.join(SAVE_DIR, 'reports', f'{name}_report.txt'), 'w', encoding='utf-8') as f:
        f.write(rep)

### D6b. "Yakshagana vs Not Yakshagana" report and threshold tuning
This answers the new question directly: **how well does each model reject non-Yakshagana photos, without rejecting real Yakshagana ones?**

* **Rejection recall**: of the true non-Yakshagana test images, how many were correctly rejected.
* **False-reject rate**: of the true Yakshagana test images, how many were wrongly rejected as "not Yakshagana".
* **Vesha accuracy (accepted)**: among true Yakshagana images that were accepted, how often the Vesha type is right.

The **threshold sweep uses the VALIDATION set only** (the test set stays untouched). Pick the threshold that suits you (stricter rejection = fewer false accepts but more false rejects) and set `OTHER_THRESHOLD` in the config cell. The final test numbers below use the current `cfg.OTHER_THRESHOLD`.


In [ ]:
OTHER = ARTIFACTS.get('other_class')
if OTHER is None:
    print('No NotYakshagana class in this model: skipping.')
else:
    oi = vesha_classes.index(OTHER)
    vi = np.array([i for i in range(len(vesha_classes)) if i != oi])

    def gate_eval(frame, prob, thr):
        true_other = (frame['vesha'].values == OTHER)
        pred_other = prob[:, oi] >= thr
        n_o, n_y = int(true_other.sum()), int((~true_other).sum())
        rej_recall = float((pred_other & true_other).sum() / n_o) if n_o else float('nan')
        false_rej = float((pred_other & ~true_other).sum() / n_y) if n_y else float('nan')
        acc_mask = (~true_other) & (~pred_other)
        ves_pred = vi[np.argmax(prob[:, vi], axis=1)]
        ves_acc = float((ves_pred[acc_mask] == frame['vesha_id'].values[acc_mask]).mean()) if acc_mask.any() else float('nan')
        return dict(rejection_recall=rej_recall, false_reject_rate=false_rej, vesha_acc_accepted=ves_acc,
                    n_not_yak=n_o, n_yak=n_y)

    # --- threshold sweep on VALIDATION (best model only) ---
    p_val = np.asarray(get_model(BEST_MODEL).predict(val_ds, verbose=0))
    sweep = pd.DataFrame({t: gate_eval(val_df, p_val, t) for t in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]}).T
    sweep.index.name = 'threshold'
    print(f'Validation sweep for {BEST_MODEL} (current cfg.OTHER_THRESHOLD = {cfg.OTHER_THRESHOLD}):')
    display(sweep[['rejection_recall', 'false_reject_rate', 'vesha_acc_accepted']].round(3))

    # --- final TEST numbers at the configured threshold, every model ---
    rows = []
    for name in trained:
        g = gate_eval(test_df, TEST[name]['prob'], cfg.OTHER_THRESHOLD)
        rows.append({'Model': name, **{k: g[k] for k in ['rejection_recall', 'false_reject_rate', 'vesha_acc_accepted']}})
    gate_tbl = pd.DataFrame(rows).set_index('Model')
    gate_tbl.to_csv(os.path.join(SAVE_DIR, 'test_yakshagana_vs_not.csv'))
    print(f'\nTEST set, threshold {cfg.OTHER_THRESHOLD} ({g["n_not_yak"]} non-Yakshagana and {g["n_yak"]} Yakshagana test images):')
    display(gate_tbl.round(3))
    if g['n_not_yak'] < 20:
        print('⚠️ Very few NotYakshagana test images: these rates are noisy. Add more negatives (especially look-alike dance forms).')

### D7. Limitations: read before trusting the numbers
* **Small datasets give noisy scores.** With 5 test images in a class, one mistake moves that class's recall by 20 points. Differences of a few points between models may be meaningless. Consider repeating with other split seeds or cross-validation.
* **Overfitting risk is high** with few images per Vesha. Watch the train-vs-validation gap in the curves.
* **Leakage can still happen.** Near-duplicate hashing catches almost-identical frames, not "same performer, different photo". Without performer folders or a `GROUP_CSV`, the test score can be optimistic. Accuracy on *new* performances, stages, cameras and lighting is likely lower.
* **Some Vesha types look alike** (shared crowns, costumes or makeup elements); some confusion is inherent, not just a model flaw.
* **The "NotYakshagana" class is only as good as its examples.** It rejects the *kinds* of non-Yakshagana images it was trained on. A dance form or scene that never appeared in `NotYakshagana/` (for example Theyyam if you only added Bharatanatya) may still be forced into a Vesha type. Add more varied, look-alike negatives and re-check D6b. Even so, the model cannot name what a rejected image shows.
* **Class imbalance:** class weights help but cannot replace more data for rare classes.
* **Hyper-parameters** (learning rates, layers unfrozen, epochs) were not tuned per model; a different setting could change the ranking.

## Part E — Load saved models (no retraining)
Reads `class_mapping.json` and rebuilds the best model from its saved weights. Works right after training **and** in a brand-new session (run Part A, Part B, then this cell). To use another saved model, set `MODEL_OVERRIDE`.


In [ ]:
MODEL_OVERRIDE = None   # e.g. 'MobileNetV2'; None -> best model recorded in class_mapping.json

load_artifacts()
available = [m for m in ARTIFACTS['models'] if os.path.exists(model_weights_path(m))]
if not available:
    raise FileNotFoundError(f'No saved weights found in {SAVE_DIR}.')
PREDICTION_MODEL = MODEL_OVERRIDE or cfg.PREDICTION_MODEL or ARTIFACTS['best_model'] or available[0]
if PREDICTION_MODEL not in available:
    raise ValueError(f'{PREDICTION_MODEL} has no saved weights. Available: {available}')
cfg.PREDICTION_MODEL = PREDICTION_MODEL
get_model(PREDICTION_MODEL)

print('Saved models found       :', available)
print(f'Model used for prediction: {PREDICTION_MODEL}'
      + ('  (best on the validation set)' if PREDICTION_MODEL == ARTIFACTS['best_model'] else '  (manually chosen)'))
print('Classes                  :', ARTIFACTS['classes'])
print('Not-Yakshagana class     :', ARTIFACTS['other_class'] or 'NONE (this model cannot reject non-Yakshagana photos)')

## Part F — Try your own image
Run the cell and use the **Choose Files** button. You can upload Yakshagana photos **and** other photos (Bharatanatya, a crowd, a temple...).

* ❌ **Not related to Yakshagana**: the model thinks the photo is not Yakshagana (`P(NotYakshagana) >= OTHER_THRESHOLD`).
* ✅ **Yakshagana + Vesha type**: relatively confident, but still only a suggestion.
* ⚠️ **Uncertain**: low confidence, a small margin between top choices, a grey-zone chance of being non-Yakshagana, or a poor-quality image.

If the model was trained **without** a NotYakshagana folder it cannot reject anything, and the output says so.


In [ ]:
LAST_IMAGE = None

def run_prediction_on_bytes(fname, data):
    global LAST_IMAGE
    try:
        img = load_rgb_image(io.BytesIO(data))
    except Exception as e:
        print(f'❌ "{fname}" could not be read as an image ({type(e).__name__}). Upload a JPG, PNG, BMP, GIF, WebP or TIFF file.')
        return
    LAST_IMAGE = img
    print(f'\n{"-" * 70}\nFile: {fname}  ({img.size[0]}x{img.size[1]} px)')
    try:
        res = predict_image(img)
    except Exception as e:
        print('❌ Prediction failed:', e); return
    show_prediction(res, img, title=fname)

if IN_COLAB:
    from google.colab import files
    print('Choose one or more images...')
    uploaded = files.upload()
    if not uploaded:
        print('No file selected. Re-run this cell to try again.')
    for fname, data in uploaded.items():
        run_prediction_on_bytes(fname, data)
else:
    print('The upload widget only works in Google Colab. Elsewhere, call run_prediction_on_bytes(name, open(path, "rb").read()).')

## Part G — Optional: low-light robustness test
Stage performances are often dimly lit. This section simulates darker, flatter photos with **brightness** and **contrast** factors (`1.0` = unchanged; smaller = darker / flatter).

* **G1** tries the settings on the image you uploaded in Part F.
* **G2** (optional) measures how the chosen model's **test** performance changes under the same settings. It needs the dataset from Part C in memory, so it is skipped in a fresh session. The model is already chosen, so this does not affect selection.

Simulated darkening only approximates real low-light photography (no sensor noise or colour casts), so treat it as a rough indicator.

In [ ]:
LOW_LIGHT_SETTINGS = [(1.0, 1.0), (0.7, 1.0), (0.5, 0.9), (0.35, 0.8), (0.25, 0.7), (0.5, 1.4)]   # (brightness, contrast)

if LAST_IMAGE is None:
    print('Upload an image in Part F first (or set LAST_IMAGE manually).')
else:
    rows = []
    n_set = len(LOW_LIGHT_SETTINGS)
    fig, axes = plt.subplots(1, n_set, figsize=(3.2 * n_set, 4.0), squeeze=False)
    for i, (b, c) in enumerate(LOW_LIGHT_SETTINGS):
        res = predict_image(LAST_IMAGE, brightness=b, contrast=c)
        axes[0, i].imshow(adjust_image(LAST_IMAGE, b, c)); axes[0, i].axis('off')
        axes[0, i].set_title(f'bright {b} / contr {c}\n{res["label"]} ({res["confidence"]:.0%})'
                             + ('\n⚠️ uncertain' if res['uncertain'] else ''), fontsize=8)
        rows.append({'brightness': b, 'contrast': c, 'Predicted Vesha': res['label'],
                     'Confidence': res['confidence'], 'Uncertain': res['uncertain']})
    plt.tight_layout(); plt.show()
    display(pd.DataFrame(rows).round(3))
    print('If the prediction changes as the image darkens, the model is sensitive to low light. '
          'Consider adding darker training photos or stronger brightness augmentation.')

In [ ]:
# G2 (optional): effect of low light on the TEST set for the model used in Part F
RUN_LOW_LIGHT_TEST_SET = True

if not RUN_LOW_LIGHT_TEST_SET:
    print('Skipped (RUN_LOW_LIGHT_TEST_SET = False).')
elif 'test_df' not in globals():
    print('Dataset not in memory (fresh session): skipping G2. Run Part C to enable it.')
else:
    model = get_model(PREDICTION_MODEL)
    rows = []
    for b, c in LOW_LIGHT_SETTINGS:
        ds = make_dataset(test_df, training=False).map(lambda x, y: (adjust_tensor(x, b, c), y), num_parallel_calls=AUTOTUNE)
        m, _ = head_metrics(test_df['vesha_id'].values, np.asarray(model.predict(ds, verbose=0)))
        rows.append({'brightness': b, 'contrast': c, 'Accuracy': m['acc'], 'Macro F1': m['f1']})
    lt = pd.DataFrame(rows)
    display(lt.round(4))
    labels = [f'b{b}/c{c}' for b, c in LOW_LIGHT_SETTINGS]
    plt.figure(figsize=(9, 4))
    plt.plot(labels, lt['Accuracy'], marker='o', label='Accuracy')
    plt.plot(labels, lt['Macro F1'], marker='o', label='Macro F1')
    plt.ylim(0, 1); plt.title(f'{PREDICTION_MODEL}: test performance under simulated low light')
    plt.xticks(rotation=30); plt.legend(); plt.grid(alpha=.3); plt.show()

## ✅ Wrap-up and next steps
* Everything needed to reuse the models is in your Drive folder: `<Model>.weights.h5`, `class_mapping.json` (class list **and** the NotYakshagana class), `split.csv`, `test_comparison.csv`, `test_yakshagana_vs_not.csv`, per-model `_meta.json` and `reports/`.
* **To improve results:** add more images for weak Vesha classes; add **look-alike negatives** (other dance and drama forms) to `NotYakshagana/`; add performer folders (or a `GROUP_CSV`) for a trustworthy split; check the confusion matrices and D6b; try several split seeds; add low-light photos to training if Part G shows a big drop.
* **Ideas for later:** temperature-scale the probabilities, add Grad-CAM, try a stronger backbone, and build a small web demo (see the project suggestions you were given).


### Final backup (includes the best-model choice from D3)

In [ ]:
# BACKUP: zip the saved models and download them to your computer (safety net)
import shutil
zip_base = os.path.join(cfg.WORK_DIR, 'yakshagana_vesha_models')
shutil.make_archive(zip_base, 'zip', SAVE_DIR)
print('Models in', SAVE_DIR, ':', sorted(os.listdir(SAVE_DIR)))
print('Backup zip:', zip_base + '.zip', f'({os.path.getsize(zip_base + ".zip") / 1e6:.0f} MB)')
if IN_COLAB:
    from google.colab import files
    files.download(zip_base + '.zip')
